# C3-gradient-descent — Session 2: Learning Rates and Stochasticity

*One class session, roughly 85 minutes. Builds directly on Session 1 (the
descent step, error factors on quadratic bowls) and on the C2-linear-models
forms: $\hat y_i = \sum_k X_{ik} w_k + b$, $\;L = \frac{1}{n}\sum_i (\hat y_i - y_i)^2$.*

**This session:** first the learning rate gets a systematic treatment — a
sweep across $\eta$ values on a known bowl.
Then the unit's payoff: gradient descent actually *fits* the C2 linear
model, end to end, on data.
Finally, stochastic gradients: paying for cheap steps with noise, on
purpose, and managing that noise with the learning rate.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 20260804

## 1. Sweeping the Learning Rate

**Motivation.**
Session 1 examined single runs.
The practical question is comparative: across *many* values of $\eta$, what
does the endpoint look like?
On $L(w) = (w - 3)^2$ we can afford a full sweep — 60 steps from $w_0 = 0$
for each of eight learning rates — and we know from the error factor
$1 - 2\eta$ exactly what to expect.

Read the printed table against three regimes:

- **Crawling** ($\eta = 0.01$): factor $0.98$; after 60 steps the error is
  still $8.93 \times 10^{-1}$ — most of the distance remains.
- **Converged to float resolution** ($\eta = 0.3$, $0.49$, $0.6$): factors
  $0.4$, $0.02$, $-0.2$; the remaining error is so small it prints as
  $0.00$, indistinguishable from zero in float64.
- **Diverged** ($\eta = 1.05$): factor $-1.1$; the "final error" is
  $9.13 \times 10^{2}$ — three hundred times the starting distance.

And one symmetry worth noticing: $\eta = 0.1$ and $\eta = 0.9$ print the
*same* error $4.60 \times 10^{-6}$, because their factors $0.8$ and $-0.8$
have equal magnitude — one crawls monotonically, the other overshoots every
step, and they land equally close after 60 steps.

In [ ]:
etas = [0.01, 0.05, 0.1, 0.3, 0.49, 0.6, 0.9, 1.05]
final_errs = []
print("  eta    factor 1-2*eta    |w_60 - 3|")
for eta in etas:
    w = 0.0
    for _ in range(60):
        w = w - eta * 2 * (w - 3.0)
    final_errs.append(abs(w - 3.0))
    print(f" {eta:5.2f}      {1 - 2 * eta:6.2f}        {abs(w - 3.0):9.2e}")

In [ ]:
plt.figure(figsize=(6, 3.4))
plt.semilogy(etas, np.maximum(final_errs, 1e-17), marker="o")
plt.axvline(0.5, color="C2", linestyle="--", linewidth=0.9, label="factor = 0 at eta = 0.5")
plt.axvline(1.0, color="C3", linestyle="--", linewidth=0.9, label="divergence at eta = 1")
plt.xlabel("learning rate eta")
plt.ylabel("|error| after 60 steps (log scale)")
plt.title("Learning-rate sweep on $L(w) = (w-3)^2$ (errors clipped at 1e-17 for plotting)")
plt.legend()
plt.show()

The U-shape (on its log scale) is the sweep signature you should expect
whenever you tune $\eta$: slow progress on the far left, a broad comfortable
basin in the middle — its floor at the factor-zero point $\eta = 0.5$, where
a single step lands exactly on the minimum — and a cliff on the right at the
divergence threshold.
On real losses nobody hands you the thresholds; a coarse sweep like this
finds the basin empirically (you will do exactly that in p10 and p18).

### Checkpoint 1

1. From the printed table: which learning rates finished with an error
   below float resolution, and why does $0.49$ beat $0.1$ so decisively?
2. Predict (no code) where $\eta = 0.55$ would land in the table, using its
   factor.
3. Why do $\eta = 0.1$ and $\eta = 0.9$ print identical errors, and in what
   *visible* way do the two runs nevertheless differ?

## 2. The Payoff: Fitting the C2 Linear Model End to End

**Motivation.**
Everything is now on the table.
As covered in C2-linear-models, the model and loss are
$$\hat y_i = \sum_{k} X_{ik} w_k + b, \qquad
L(w, b) = \frac{1}{n} \sum_{i=1}^{n} (\hat y_i - y_i)^2,$$
and F4's sum-of-squares chain rule gives the gradients
$$\frac{\partial L}{\partial w_k} = \frac{2}{n} \sum_{i} (\hat y_i - y_i)\, X_{ik},
\qquad
\frac{\partial L}{\partial b} = \frac{2}{n} \sum_{i} (\hat y_i - y_i).$$
C2 could only *evaluate* these at given parameters.
Today we close the loop: start at $w = \mathbf{0}$, $b = 0$ and let the
descent step choose the parameters.

The data below is generated from known parameters
$w^\ast = (1.5, -2.0, 0.5)$, $b^\ast = 0.8$ plus Gaussian noise
(sd $0.3$), so we can grade the fit afterwards.

In [ ]:
rng = np.random.default_rng(SEED)
n, d = 200, 3
X = rng.normal(0, 1, (n, d))
w_star = np.array([1.5, -2.0, 0.5])
b_star = 0.8
y = (X * w_star).sum(axis=1) + b_star + rng.normal(0, 0.3, n)
print("X shape:", X.shape, "  y shape:", y.shape)

In [ ]:
def mse_loss(X, y, w, b):
    errors = (X * w).sum(axis=1) + b - y          # (n,) residuals  yhat_i - y_i
    return (errors ** 2).mean()


def mse_gradients(X, y, w, b):
    """The pinned C2 gradient formulas — broadcasting and axis sums only."""
    errors = (X * w).sum(axis=1) + b - y                      # (n,)
    grad_w = 2 / len(y) * (errors[:, None] * X).sum(axis=0)   # (d,)
    grad_b = 2 / len(y) * errors.sum()
    return grad_w, grad_b


# trust, but verify (F4 style): central differences at the start point
w0, b0, h = np.zeros(d), 0.0, 1e-6
gw, gb = mse_gradients(X, y, w0, b0)
num = np.zeros(d)
for j in range(d):                                # a checker MAY loop
    step = np.zeros(d)
    step[j] = h
    num[j] = (mse_loss(X, y, w0 + step, b0) - mse_loss(X, y, w0 - step, b0)) / (2 * h)
num_b = (mse_loss(X, y, w0, b0 + h) - mse_loss(X, y, w0, b0 - h)) / (2 * h)
print("gradient check gap:", f"{max(np.abs(num - gw).max(), abs(num_b - gb)):.1e}")

The checker agrees to about $10^{-9}$ — the formulas are trustworthy.
Now the algorithm.
Note the shape of the code: the *gradient* is pure broadcasting (no loops
over data), while the *iteration* is an honest `for` loop over steps —
that loop is the algorithm, not a style violation.

In [ ]:
def fit(X, y, eta, steps):
    w = np.zeros(X.shape[1])
    b = 0.0
    losses = []
    for _ in range(steps):                       # THE loop: iteration is the algorithm
        grad_w, grad_b = mse_gradients(X, y, w, b)
        w = w - eta * grad_w
        b = b - eta * grad_b
        losses.append(mse_loss(X, y, w, b))
    return w, b, np.array(losses)


print(f"loss before any step: {mse_loss(X, y, np.zeros(d), 0.0):.4f}")
w_fit, b_fit, losses = fit(X, y, eta=0.1, steps=400)
for t in (1, 50, 100, 400):
    print(f"loss after step {t:3d}: {losses[t - 1]:.8f}")
print()
print("learned w:", np.round(w_fit, 4), "  learned b:", round(b_fit, 4))
print("true    w:", w_star, "        true    b:", b_star)

In [ ]:
plt.figure(figsize=(6, 3.2))
plt.semilogy(losses)
plt.xlabel("step")
plt.ylabel("L(w, b)  (log scale)")
plt.title("The payoff: MSE under gradient descent, eta = 0.1")
plt.show()

Read the trace:

- the loss falls from $6.3067$ to $4.2516$ in one step, is at
  $0.07700629$ by step 50, and from step 100 on repeats
  $0.07700625$ — flat to the printed precision;
- the learned parameters $(1.4584, -1.9778, 0.4948)$, $b = 0.8006$ sit
  within about $0.04$ of the generating values — as close as this noisy
  sample permits;
- the floor $\approx 0.077$ is **not** a failure to optimize: the data was
  generated with noise of variance $0.3^2 = 0.09$, and no parameter choice
  can undo that randomness. The floor *is* (this sample's version of) the
  noise.

**When to stop.**
The loss curve is the instrument: stop when it flattens (the improvement per
step falls below some tolerance), or after a fixed budget of steps —
both criteria are used in practice, and the curve above satisfies either by
step $\sim$100.

### Checkpoint 2

1. Why does $\partial L / \partial b$ carry no $X_{ik}$ factor, while
   $\partial L / \partial w_k$ does? (Differentiate $\hat y_i$.)
2. The trace flattens at $0.077$ rather than $0$. Why is that the expected
   outcome here, and what loss floor would you expect if the noise sd had
   been $0.1$?
3. With $\eta = 0.001$ instead: what changes about the trace, and what
   stays the same eventually?

## 3. Mini-Batches: Noisy but Cheap

**Motivation.**
Each full gradient touches all $n$ examples.
At $n = 200$ that is free; at $n = 10^7$ it makes every single step a pass
over the whole dataset.
The fix is to *estimate* the gradient from a **mini-batch**: a small random
sample of the data (F5's vocabulary, on purpose — a batch is a random
sample, and a batch gradient is a sample statistic).

**Definition.**
Pick batch size $B$ and draw indices $i_1, \dots, i_B$ uniformly at random
(seeded, as always) from $\{0, \dots, n-1\}$.
The **mini-batch gradient** applies the pinned formulas to those $B$
examples only — the same code, handed `X[idx]` and `y[idx]` with the mean
taken over $B$.
Descent that uses batch gradients is **stochastic gradient descent**; one
full-data pass' worth of examples ($n/B$ batches) is called an **epoch**.

**The one theorem that makes this sane** (proved as p12): with uniformly
chosen indices,
$$E[\,\text{batch gradient}\,] = \text{full gradient}.$$
Each batch gradient is wrong, but *unbiased* — right on average, in the F5
expectation sense.
Watch the spread at a fixed parameter point:

In [ ]:
w_probe = np.array([1.0, -1.0, 0.0])
b_probe = 0.0
g_full, gb_full = mse_gradients(X, y, w_probe, b_probe)

rng_batch = np.random.default_rng(SEED)
demo_idx = rng_batch.integers(0, n, size=(12, 10))     # 12 batches of size 10
batch_gs = []
for row in demo_idx:                                   # loop over batches: allowed
    g, _ = mse_gradients(X[row], y[row], w_probe, b_probe)
    batch_gs.append(g)
batch_gs = np.array(batch_gs)

print("full gradient, first component   :", f"{g_full[0]:7.3f}")
print("12 batch gradients, same component:", np.round(batch_gs[:, 0], 3))
print(f"their mean: {batch_gs[:, 0].mean():7.3f}   (drifting toward the full value, but 12 is few)")

The twelve size-10 estimates of the first gradient component scatter widely
— from about $-1.5$ to $+0.16$ around the full value $-0.813$ — and their
mean ($-0.547$) sits tighter than the typical single estimate, though a
dozen batches is far from enough to pin the value down.
That is exactly the story F5 told about sample means: individual samples are
noisy, averages of many concentrate (p08 averages 2000 of them and lands
within $0.03$).
The bargain: each of those estimates cost $10$ example-evaluations instead
of $200$ — a twentieth of the price, for a still-useful direction.

### Checkpoint 3

1. For a batch of size 1 (a single uniformly chosen example $i$): what is
   the expected value of its gradient $\nabla \ell_i$, and which F5 fact
   delivers the answer in one line?
2. At $n = 10^6$ and $B = 32$: how many times cheaper is a batch step than
   a full step, and how many steps make up one epoch?

## 4. Full vs Stochastic: the Seeded Head-to-Head

Same data, same $\eta = 0.1$, same 400 steps: full-batch descent against
stochastic descent with batches of 10.
Both runs record the **full-data** loss after every step (the honest score —
Section 6's Pitfall 4 is about what happens if you don't).
The batch sequence is seeded, so the "random" run is exactly repeatable.

In [ ]:
def fit_sgd(X, y, eta, steps, batch_size, seed):
    rng_local = np.random.default_rng(seed)
    batch_rows = rng_local.integers(0, len(y), size=(steps, batch_size))
    w = np.zeros(X.shape[1])
    b = 0.0
    losses = []
    for t in range(steps):                       # the step loop, as before
        idx = batch_rows[t]
        grad_w, grad_b = mse_gradients(X[idx], y[idx], w, b)
        w = w - eta * grad_w
        b = b - eta * grad_b
        losses.append(mse_loss(X, y, w, b))      # scored on ALL the data
    return w, b, np.array(losses)


w_sgd, b_sgd, losses_sgd = fit_sgd(X, y, eta=0.1, steps=400, batch_size=10, seed=SEED)

print(f"full-batch final loss : {losses[-1]:.6f}   (200 examples touched per step)")
print(f"stochastic final loss : {losses_sgd[-1]:.6f}   (10 examples touched per step)")
print("sgd learned w:", np.round(w_sgd, 4), "  b:", round(b_sgd, 4))

In [ ]:
plt.figure(figsize=(6.5, 3.4))
plt.semilogy(losses, label="full batch (200/step)")
plt.semilogy(losses_sgd, label="stochastic, batch 10", alpha=0.8)
plt.xlabel("step")
plt.ylabel("full-data loss (log scale)")
plt.title("Full vs stochastic descent, same eta = 0.1, seeded")
plt.legend()
plt.show()

Three observations, all visible in the plot and the prints:

- **Quality:** the stochastic run finishes at $0.083864$ vs the full run's
  $0.077006$ — close, but sitting on a slightly raised, jittery floor.
  The noisy steps never let the iterate rest exactly at the bottom.
- **Cost:** each stochastic step touched $10$ examples, each full step
  $200$. After 20 stochastic steps the run has consumed one full-batch
  step's worth of data (200 examples) — and its full-data loss is already
  down to $0.0852$, while the full-batch run's single step only reached
  $4.2516$. Per example touched, the noisy version is *far* ahead early.
- **Texture:** the stochastic curve is not monotone. Individual steps go
  uphill; the *trend* goes down. That is what "right on average" buys.

### Checkpoint 4

1. From the printed numbers: after equal *data budgets* (one full pass'
   worth), which run has the lower full-data loss, and by roughly how much?
2. The stochastic curve rises on some steps. Why is that not a bug, and
   which quantity would you inspect if you suspected it *were* one?

## 5. Step Size Meets Noise: the Floor

**Motivation.**
For full-batch descent on a bowl, smaller $\eta$ just means slower.
For *stochastic* descent, $\eta$ takes on a second job: it scales the noise.
Every batch gradient equals the full gradient plus a random error; the step
passes that error to the parameters multiplied by $\eta$.
Near the minimum the useful signal shrinks (the full gradient $\to 0$) but
the noise does not — so the iterate ends up rattling around the bottom in a
band whose width grows with $\eta$.

Three stochastic runs, identical seeded batches, only $\eta$ varies.
The **floor** — the average full-data loss over the last 100 steps — tells
the story:

In [ ]:
print("  eta     final loss    floor (mean of last 100)")
for eta in (0.02, 0.1, 0.3):
    _, _, ls = fit_sgd(X, y, eta=eta, steps=400, batch_size=10, seed=SEED)
    print(f" {eta:5.2f}    {ls[-1]:.6f}      {ls[-100:].mean():.6f}")
print(f"  (full-batch reference floor: {losses[-1]:.6f})")

The floors line up with the theory: $\eta = 0.02$ settles to $0.0778$ —
almost the full-batch floor of $0.0770$ — while $\eta = 0.1$ rattles at
$0.0813$ and $\eta = 0.3$ at $0.0929$.
The ranking of what you *pay* is the mirror image: the small-$\eta$ run
crawls early exactly as Section 1's sweep predicts.

This tension has a standard resolution, which you will build yourself in
p18: **start large, then shrink** — a decaying learning-rate schedule rides
the fast early progress of a big step, then lowers the floor by shrinking
the noise multiplier.
(Generic idea, no special machinery: multiply $\eta$ by $\tfrac12$ every so
many steps.)

### Checkpoint 5

1. Order the three floors printed above and state the rule they illustrate
   in one sentence.
2. Why does the *full-batch* run have no comparable $\eta$-dependent floor
   (for any stable $\eta$)?

## 6. Worked Exam-Style Example 2: Constrained Coding

The register the real paper uses for implementation items: an exact function
contract, a shape contract, and an API ban list with a zero-points clause.
Worked fully, then verified.

---

**Problem.**
Given `X` of shape `(n, d)` and `y` of shape `(n,)`, implement

`fit_mse(X, y, eta, steps)` → returns `(w, b, losses)`

running exactly `steps` gradient-descent updates of the mean-MSE loss of
the model $\hat y_i = \sum_k X_{ik} w_k + b$, from `w = np.zeros(d)`,
`b = 0.0`, recording the loss after each update in `losses` (shape
`(steps,)`).

**Banned (zero points): `@`, `np.matmul`, `np.dot`, `np.einsum`, `.T`.**
The only permitted loop is the loop over the `steps` updates; gradients and
losses must use broadcasting and axis reductions only.

*Grading data: a deterministic 4-point set with an exact linear relation
$y = 2 x_1 - x_2 + 1$, so a correct implementation must drive the loss to
(numerically) zero and recover $w = (2, -1)$, $b = 1$.*

---

**Solution, step by step.**

*Step 1 — residuals with broadcasting.* `(X * w).sum(axis=1) + b - y` is the
shape-`(n,)` residual vector $\hat y - y$; `X * w` broadcasts `(n, d) * (d,)`
row-wise, and the `axis=1` sum realizes $\sum_k X_{ik} w_k$ without `@`.

*Step 2 — the two gradients.* Weights:
`2 / n * (errors[:, None] * X).sum(axis=0)` — the `(n, 1) * (n, d)` product
puts $e_i X_{ik}$ in cell $(i, k)$, and the `axis=0` sum contracts over
examples. Bias: `2 / n * errors.sum()`.

*Step 3 — the loop.* Exactly `steps` iterations of the Session 1 update,
appending the post-update loss each time.

*Step 4 — verify before trusting:* central-difference gradient check, loss
monotonically decreasing (full-batch on a bowl must be), and recovery of the
known parameters.

In [ ]:
def fit_mse(X, y, eta, steps):
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0
    losses = []
    for _ in range(steps):                                    # the only loop
        errors = (X * w).sum(axis=1) + b - y                  # (n,)
        grad_w = 2 / n * (errors[:, None] * X).sum(axis=0)    # (d,)
        grad_b = 2 / n * errors.sum()
        w = w - eta * grad_w
        b = b - eta * grad_b
        losses.append((((X * w).sum(axis=1) + b - y) ** 2).mean())
    return w, b, np.array(losses)


X_test = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [2.0, 1.0]])
y_test = np.array([3.0, 0.0, 2.0, 4.0])                       # exactly 2*x1 - x2 + 1

w_out, b_out, ls = fit_mse(X_test, y_test, eta=0.2, steps=600)
print("learned w:", np.round(w_out, 6), "  b:", f"{b_out:.6f}")
print("final loss:", f"{ls[-1]:.1e}")
print("loss decreased every step:", bool(np.all(np.diff(ls) <= 0)))

Output check: $w = (2, -1)$, $b = 1.000000$, final loss at the $10^{-23}$
scale — zero for float purposes — and the monotone-decrease flag is `True`.
On the real exam, that last flag is a free self-test: a full-batch descent
whose recorded losses ever increase (on a quadratic loss, with a stable
$\eta$) has a bug or a bad learning rate, full stop.

### Checkpoint 6

1. State precisely why the step loop survives the ban list but a loop over
   the $n$ examples would not.
2. As covered in C2-linear-models, adding an L2 penalty
   $\lambda \sum_k w_k^2$ changes the loss. What single line of `fit_mse`
   changes, and to what?

## 7. Common Pitfalls II

**Pitfall 4 — judging progress by the batch loss.**
The batch loss is a *sample estimate* of the full loss.
At completely fixed parameters, it jumps around from batch to batch — so a
batch loss that rose between two steps is (on its own) evidence of nothing.
Below: the fitted parameters from Section 2 held perfectly still, eight
different batches scored:

In [ ]:
rng_demo = np.random.default_rng(SEED)
rows = rng_demo.integers(0, n, size=(8, 10))
batch_losses = []
for r in rows:
    e = (X[r] * w_fit).sum(axis=1) + b_fit - y[r]
    batch_losses.append((e ** 2).mean())
print("full-data loss at fixed (w, b):", f"{mse_loss(X, y, w_fit, b_fit):.4f}")
print("eight batch losses, same (w, b):", [f"{v:.4f}" for v in batch_losses])

One frozen model, batch scores from $0.0297$ to $0.0917$ — a three-fold
spread around the true $0.0770$.
The fix: make decisions on the **full-data loss** (or at least a large,
*fixed* subset scored consistently), never on the loss of whatever batch
happened to be drawn.

**Pitfall 5 — losing the seed.**
A stochastic run is an experiment; without a seed it is an unrepeatable one,
and "it worked yesterday" becomes undebuggable.
Seeds make noise reproducible:

In [ ]:
a = np.random.default_rng(20260804).integers(0, 200, 10)
b_ = np.random.default_rng(20260804).integers(0, 200, 10)
c = np.random.default_rng(12345).integers(0, 200, 10)
print("same seed, same batches      :", bool(np.array_equal(a, b_)))
print("different seed, same batches :", bool(np.array_equal(a, c)))

**Pitfall 6 — sum over the batch (the $n$-vs-$B$ relapse).**
Session 1's Pitfall 3 returns wearing a batch: if the batch gradient uses
`.sum()` where the full gradient used `.mean()`-style $\frac1n$ scaling,
the gradient magnitude silently depends on the batch size — and the same
$\eta$ that was stable at $B = 10$ can explode when someone "just" raises
the batch size.
With the pinned mean forms, gradients from a batch of 10 and from all 200
examples live on the same scale:

In [ ]:
r = rows[0]
e = (X[r] * w_fit).sum(axis=1) + b_fit - y[r]
g_mean = 2 / 10 * (e[:, None] * X[r]).sum(axis=0)   # pinned: mean over the batch
g_sum = 2 * (e[:, None] * X[r]).sum(axis=0)         # BROKEN: sum over the batch
print("first components — mean form:", f"{g_mean[0]:.4f}", "  sum form:", f"{g_sum[0]:.4f}")
print("ratio sum/mean:", f"{g_sum[0] / g_mean[0]:.1f}", "  (= the batch size)")

### Checkpoint 7

1. A teammate reports "the loss went up between steps 400 and 401, so the
   optimizer is broken" — they are reading the current batch's loss.
   Give the two-sentence response, citing what Pitfall 4's demo showed.
2. Two runs of "the same" stochastic training script produce different
   final parameters. What is the first thing to check, and what should the
   script have contained?

## Exam Connections

How this unit's material shows up in Round 1 (paraphrased from the
`reference/analysis.md` topic table — no real test text here):

- The **NumPy implementation cluster** (8 sub-parts, 55 points in r1-2026)
  is built around broadcasting-only gradient code with explicit API bans and
  zero-point clauses — Section 6's register.
  A descent loop wrapped around a banned-API-free gradient is precisely the
  texture of p05, p07, p09, and p14.
- The **ML concepts cluster** (5 concept MCQs, 50 points) asks
  reasoning-free conceptual questions in the style of "which learning rate
  produced this behavior" — the register of p01–p03 and the Section 1
  sweep; the analysis's difficulty profile counts these among the most
  reachable points on the paper.
- The paper's **multi-part arcs** make later parts consume earlier results
  (a derivation feeds an implementation feeds a checker feeds a
  conclusion); p13, p14, and p18 train exactly that scaffolding, with the
  normal-form arithmetic of p04 as the hand-computation atom.
- The r2-2026 rationale material singles out *training-dynamics reasoning*
  (why a run diverged, what a plateau means) as a discriminator — Sections
  1, 5, and 7 are that skill, drilled with numbers.

## Going Deeper

Optional forward pointers along the course map — nothing here is needed for
this unit's practice:

- **`C5-neural-networks`**: layered models replace the linear $\hat y$, and
  their gradients come from F4's chain rule applied route by route — but
  the training loop you built in Section 2 survives *unchanged*: loss,
  gradient, step, repeat.
  Learning-rate discipline and loss-curve reading transfer verbatim.
- **`C6-pytorch`**: a framework computes the gradients automatically; what
  remains yours is exactly this unit's content — the update rule, the
  learning rate, the batch size, and the judgment about what the loss curve
  is saying.

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. $\eta \in \{0.3, 0.49, 0.6\}$ print $0.00$. Factors: $0.49$ gives
   $|1 - 2\eta| = 0.02$, vs $0.8$ for $\eta = 0.1$ — per step it wipes out
   $98\%$ of the error instead of $20\%$, and $0.02^{60}$ underflows any
   float.
2. Factor $1 - 1.1 = -0.1$: overshooting each step but contracting by
   $10\times$ per step — it would land in the "prints $0.00$" group.
3. Factors $0.8$ and $-0.8$ share a magnitude, so after 60 steps the error
   sizes match exactly. Visibly, the $\eta = 0.9$ run alternates sides of
   the minimum every step, while $\eta = 0.1$ approaches from one side.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. $\hat y_i = \sum_k X_{ik} w_k + b$, so
   $\partial \hat y_i / \partial w_k = X_{ik}$ but
   $\partial \hat y_i / \partial b = 1$ — the chain rule's inner factor is
   $X_{ik}$ for weights and $1$ for the bias.
2. The targets contain noise no linear model can reproduce, so the loss
   bottoms out near the noise variance ($0.3^2 = 0.09$; this sample's
   floor happens to sit at $0.077$). With sd $0.1$: a floor near
   $0.1^2 = 0.01$.
3. The descent is $\sim 100\times$ slower (factor much closer to 1 per
   step), so the trace flattens far later — but toward the *same* floor:
   $\eta$ changes the journey, not (for stable values) the destination.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. $E[\nabla \ell_I] = \sum_i \frac{1}{n} \nabla \ell_i = $ the full
   gradient — the expectation of a discrete uniform random variable
   (F5), values weighted by probability $1/n$.
2. $10^6 / 32 = 31250$ times cheaper per step; one epoch is
   $10^6 / 32 = 31250$ steps.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. After 200 examples' worth: SGD (20 steps) is at $0.0852$; full-batch
   (1 step) is at $4.2516$ — a factor of ~50 in loss for the same data
   budget.
2. Not a bug because batch gradients are only right *on average* —
   individual steps can go uphill. To check for a real bug, look at the
   *full-data* loss trend (or temporarily switch to full-batch descent,
   which must decrease monotonically for a stable $\eta$).

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. $0.0778 < 0.0813 < 0.0929$ for $\eta = 0.02 < 0.1 < 0.3$: the
   stochastic floor rises with the learning rate, because $\eta$ multiplies
   the gradient noise injected into the parameters each step.
2. Full-batch gradients carry no sampling noise: near the minimum the
   gradient itself $\to 0$, so the iterate settles instead of rattling —
   any stable $\eta$ reaches (in the limit) the same bottom.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. The ban register targets replacing *vectorizable per-example
   arithmetic* with Python loops; the step loop is not vectorizable
   arithmetic but the algorithm's sequential structure itself — each
   iterate depends on the previous one.
2. Only the `grad_w` line: it becomes
   `grad_w = 2 / n * (errors[:, None] * X).sum(axis=0) + 2 * lam * w`
   (the penalty's gradient $2\lambda w_k$ from C2-linear-models added on;
   the bias is conventionally left unpenalized).

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. "At completely fixed parameters, our eight batch scores ranged from
   $0.0297$ to $0.0917$ — batch losses jump around with no change in the
   model at all. Judge the run on the full-data loss curve; ours is
   trending down."
2. Check whether the batch sampling was seeded. The script should have
   created its generator as `np.random.default_rng(SEED)` with a recorded
   seed, making the run exactly repeatable.

</details>